# ICU Length of Stay Prediction

In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl
from datetime import timedelta

from ethos.constants import PROJECT_ROOT
from ethos.constants import SpecialToken as ST
from ethos.inference.constants import Reason, Task
from ethos.metrics import compute_and_print_metrics_regression, preprocess_inference_results
from ethos.utils import get_mimic_hf_patients, get_mimic_sepsis_icu_stays

# Requires original MIMIC-IV 2.2 dataset that we do not provide
mimic_sepsis = get_mimic_sepsis_icu_stays()
# heart_failure_cohort = get_mimic_hf_patients(PROJECT_ROOT / "data/mimic-2.2")

/home/henri/ethos-benchmark/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# DEBUGGING ONLY
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'pyproject.toml').exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.ethos.datasets.length_of_stay import LosPredictionDataset
path = "../../../../mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test"
dataset = LosPredictionDataset(path, offset=timedelta(hours=24))

Found 8637 hospital stays with an outcome token.


In [ ]:
import torch
torch.tensor(5036215500000000).item()*1e-6+timedelta(hours=12).total_seconds()

In [ ]:
import torch
time_chunk = [torch.tensor(5036215500000000)]
offset = timedelta(hours=12)
deadline = time_chunk[0] // 1_000_000 + int(offset.total_seconds())
print(deadline)

In [ ]:
5036215500000000*1e-6 + offset.total_seconds()

In [ ]:
dataset[0][1]

In [ ]:
for d in dataset:
    if d[1]['hadm_id'] == 23822395:
        print(d[1])

In [3]:
import pandas as pd
df_token = pd.read_parquet("/mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test/11_inject_time_intervals/0.parquet")

In [6]:
df_token.iloc[505:515]

,subject_id,time,code,numeric_value,text_value,hadm_id,edstay_id,icustay_id,insurance,language,marital_status,race,drg_severity,drg_mortality,emar_id,emar_seq,priority
505,10002155,2129-08-04 17:56:00,LAB//51277//%,15.20,NaN,23822395.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STAT
506,10002155,2129-08-04 17:56:00,Q7,15.20,NaN,23822395.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STAT
507,10002155,2129-08-04 17:56:00,LAB//51279//M/UL,4.19,NaN,23822395.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STAT
508,10002155,2129-08-04 17:56:00,Q7,4.19,NaN,23822395.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STAT
509,10002155,2129-08-04 17:56:00,LAB//51301//K/UL,5.50,NaN,23822395.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STAT
510,10002155,2129-08-04 17:56:00,Q3,5.50,NaN,23822395.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STAT
511,10002155,2129-08-05 05:01:00,8h-12h,1.00,NaN,23822395.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STAT
512,10002155,2129-08-05 05:01:00,LAB//51237//UNK,1.00,NaN,23822395.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STAT
513,10002155,2129-08-05 05:01:00,Q1,1.00,NaN,23822395.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STAT
514,10002155,2129-08-05 05:01:00,LAB//51274//SEC,11.90,NaN,23822395.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STAT


# MIMIC

In [ ]:
results_dir = PROJECT_ROOT / "results"
los_pred_result_dir = results_dir / Task.LOS_PREDICTION

def preprocess_los_prediction_results(input_dir) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual").is_in([ST.DEATH, ST.ICU_DISCHARGE]),
        expected_expr=pl.col("expected").is_in([ST.DEATH, ST.ICU_DISCHARGE]),
        filter_ambiguous=(
            pl.col("actual").is_in([ST.DEATH, ST.ICU_DISCHARGE])
            & (pl.col("stop_reason") == Reason.GOT_TOKEN)
        ),
    )

In [ ]:
los_pred_results = []
for result_fp in los_pred_result_dir.iterdir():
    df = preprocess_los_prediction_results(result_fp)
    rep_num = df["counts"].mean()
    plt.figure(figsize=(10, 5))
    plt.suptitle(f"Length of Stay Prediction\n{result_fp.name} (rep_num={rep_num:.2f})")
    plt.subplot(1, 2, 1)
    res_full = compute_and_print_metrics_regression(df["expected"], df["actual"], "Full")

    plt.subplot(1, 2, 2)
    # hf_df = df.filter(pl.col("patient_id").is_in(heart_failure_cohort))
    # res_hf = compute_and_print_metrics(hf_df["expected"], hf_df["actual"], "HF patients")

    los_pred_results.append(
        {
            "name": result_fp.name,
            "auc": res_full["fitted_auc"],
            # "auc_hf": res_hf["fitted_auc"],
            "rep_num": rep_num,
        }
    )
    plt.tight_layout()
    plt.show()
los_pred_results = pl.from_dicts(los_pred_results).sort("auc", descending=True)

In [ ]:
los_pred_results